In [1]:
import numpy as np

In [2]:
def neighbours(tile):
    nbrs=[tile]
    if tile>=4: nbrs.append(tile-4)
    if tile<12: nbrs.append(tile+4)
    if tile%4: nbrs.append(tile-1)
    if tile%4!=3: nbrs.append(tile+1)
    return nbrs

toggle_mask=[]

for a in range(16):
    mask=0
    for cell in neighbours(a):
        mask ^= (1<<cell)
    toggle_mask.append(mask)

In [3]:
lo_map = {}

for state in range(2**16):
    transitions={}
    for action in range(16):
        next_state = state ^ toggle_mask[action]
        reward = 1 if next_state==0 else -1
        done = (next_state==0)
        transitions[action]=[(1.0,next_state,reward,done)]
    lo_map[state]=transitions

print("MDP built with",len(lo_map),"states")

MDP built with 65536 states


In [4]:
def int_to_grid(x):
    bits=[]
    for i in range(15,-1,-1):
        bits.append((x>>i)&1)
    return bits

def print_grid(x):
    b=int_to_grid(x)
    for i in range(0,16,4):
        print(*b[i:i+4])
    print()

In [5]:
states=list(range(2**16))
actions=list(range(16))
goal_state=0

gamma=0.9
theta=1e-4

V={s:0.0 for s in states}
V[goal_state]=0.0

while True:
    delta=0
    Vnew=V.copy()

    for s in states:
        if s==goal_state:
            continue

        best=-1e18
        for a in actions:
            p,ns,r,_ = lo_map[s][a][0]
            val = p*(r + gamma*V[ns])
            best=max(best,val)

        Vnew[s]=best
        delta=max(delta,abs(V[s]-best))

    V=Vnew
    if delta < theta:
        break

print("Value Iteration Converged")

Value Iteration Converged


In [6]:
policy={}

for s in states:
    if s==goal_state:
        policy[s]=None
        continue

    best=-1e18
    besta=None

    for a in actions:
        p,ns,r,_=lo_map[s][a][0]
        val=p*(r+gamma*V[ns])
        if val>best:
            best=val
            besta=a

    policy[s]=besta

print("Policy Extracted")

Policy Extracted


In [7]:
def action_to_coord(a):
    return (a//4 + 1 , a%4 + 1)

In [8]:
# Example starting board
start_bits = [
0,0,0,1,
0,1,1,0,
0,1,0,1,
0,1,0,0
]

start_state=0
for i,b in enumerate(start_bits):
    start_state |= (b<<(15-i))

In [9]:
s=start_state
moves=0
LIMIT=100

print("Initial Board:")
print_grid(s)

while s!=0 and moves<LIMIT:
    a = policy[s]
    print("Press:",action_to_coord(a))
    s = s ^ toggle_mask[a]
    print_grid(s)
    moves+=1

if s==0:
    print("DONE")
else:
    print("FAILED")

print("Moves:",moves)

Initial Board:
0 0 0 1
0 1 1 0
0 1 0 1
0 1 0 0

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 0
0 1 1 1

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 1
0 1 0 0

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 0
0 1 1 1

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 1
0 1 0 0

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 0
0 1 1 1

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 1
0 1 0 0

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 0
0 1 1 1

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 1
0 1 0 0

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 0
0 1 1 1

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 1
0 1 0 0

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 0
0 1 1 1

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 1
0 1 0 0

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 0
0 1 1 1

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 1
0 1 0 0

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 0
0 1 1 1

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 1
0 1 0 0

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 0
0 1 1 1

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 1
0 1 0 0

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 0
0 1 1 1

Press: (1, 1)
0 0 0 1
0 1 1 0
0 1 0 1
0 1 0 0

Press: (1, 1

In [10]:
all_on = (1<<16)-1
s=all_on
steps=0

while s!=0 and steps<200:
    s ^= toggle_mask[policy[s]]
    steps+=1

print("Steps from all-ones:",steps)

Steps from all-ones: 4
